# Char vs Word: RNN vs Transformer Language Models

**Assignment:** Architectural Deep Dive — Transformers vs. RNNs  
**Goal:** Build, train, and benchmark four models (CharRNN, CharTransformer, WordRNN, WordTransformer) on a Wikipedia corpus (~55k characters).

Pipeline:
1. Setup
2. Corpus Acquisition
3. Tokenization Pipelines
4. Model Definitions
5. Training (loss + perplexity)
6. Inference & Temperature Scaling
7. Export Results (`results/pic`, `results/data`)


## 1. Setup

Import libraries, set random seed, and select device (`cuda` if available, else `cpu`).


In [ ]:
# 导入实验所需的标准库与第三方库
import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter

import requests
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

print("Imports OK")


In [ ]:
# 固定随机种子，保证实验可复现
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 自动选择 GPU（云端 Colab 通常有 cuda）或 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")


In [ ]:
# 创建输出目录：图表 -> results/pic，关键数值 -> results/data
# 云端运行时会在当前工作目录下生成这两个文件夹
PIC_DIR = Path("results/pic")
DATA_DIR = Path("results/data")
PIC_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Output dirs:", PIC_DIR.resolve(), DATA_DIR.resolve())
